# 01 | Synthetic Data Generation (Live Demo)

**Objective:** Generate realistic banking call transcripts (Fraud vs. Legitimate) for Supervised Fine-Tuning (SFT).

**NVIDIA Stack & Architecture:**
* **NVIDIA NIM (`meta/llama-3.1-8b-instruct`):** Local high-throughput inference endpoint.
* **NeMo Guardrails:** Controlled generation enforcing strict JSON output formats.
* **LLM-as-a-Judge:** Automated quality gating (only transcripts scoring $\ge 4/5$ are saved).
* **Async Concurrency:** Parallelized API execution via `asyncio`.
* **Weights & Biases (W&B) Lineage:** Configured with `job_type="data-generation"` and versioned `raw-transcripts` artifacts for end-to-end traceability.

## 1. Setup & W&B Lineage Configuration
Configured for fast live execution (`NUM_SAMPLES = 100`) with explicit W&B `job_type` and artifact lineage tracking.

In [1]:
import asyncio
import json
import os
import random
import re
import time
from typing import Tuple
from pydantic import BaseModel, Field
from openai import OpenAI
from nemo_curator import OpenAIClient
from nemoguardrails import LLMRails, RailsConfig
import wandb

# Demo Runtime Parameters
NIM_BASE_URL = os.getenv("NIM_BASE_URL", "http://172.17.0.1:8800/v1")
GEN_MODEL = "meta/llama-3.1-8b-instruct"
JUDGE_MODEL = "meta/llama-3.1-8b-instruct"

OUTPUT_DATA_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/01_Data_Generation/data/transcripts.jsonl"
OUTPUT_BENCHMARK_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/01_Data_Generation/data/benchmark.jsonl"

NUM_SAMPLES = 100        # Fast demo mode (Production: 5000)
CONCURRENCY_LIMIT = 4    # Parallel requests
ARTIFACT_NAME = "raw-transcripts"  # Standardized name for downstream Curation stage

# Initialize Weights & Biases Run with job_type for End-to-End Lineage
run = wandb.init(
    project="nemo-fraud-detection",
    job_type="data-generation",
    name="synthetic-data-generation-advanced",
    tags=["stage-1-generation", "nim", "guardrails"],
    config={
        "num_samples": NUM_SAMPLES,
        "gen_model": GEN_MODEL,
        "judge_model": JUDGE_MODEL,
        "concurrency_limit": CONCURRENCY_LIMIT,
        "min_judge_score": 4,
        "output_artifact": ARTIFACT_NAME
    }
)

base_openai_client = OpenAI(base_url=NIM_BASE_URL, api_key="not-needed")
curator_openai_client = OpenAIClient(base_openai_client)

semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
file_lock = asyncio.Lock()
completed_counter = 0
generated_records = []  # Collect records for W&B Table inspection

print(f"✅ NIM Client & W&B Lineage initialized (job_type='data-generation', Target: {NUM_SAMPLES} samples).")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.
wandb: Currently logged in as: lohmann-daniel (lohmann-daniel-myself) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


✅ NIM Client & W&B Lineage initialized (job_type='data-generation', Target: 100 samples).


## 2. Scenario Catalog & Pydantic Validation Schema
Combines randomized customer personas, conversational tones, and fraud/legitimate banking scenarios with strict schema enforcement.

In [2]:
CUSTOMER_NAMES = ["Mr. Miller", "Ms. Smith", "Mr. Taylor", "Ms. Fisher", "Mr. Weber", "Ms. Meyer"]

FRAUD_SCENARIOS = [
    "Customer demands immediate account unlock, refusing security codes claiming their phone is broken.",
    "Customer requests an online banking password reset using stolen master data, failing the security question.",
    "Customer demands an urgent wire transfer and heavily pressures the agent due to an alleged emergency.",
    "Customer claims the 2FA app is broken and tries to persuade the agent to manually authorize the TAN."
]

LEGITIMATE_SCENARIOS = [
    "Customer asks for their account balance and recent transactions from the last two weeks.",
    "Customer wants to modify the amount and execution date of an existing standing order.",
    "Customer wants to block their lost debit card and order a replacement card.",
    "Customer wants to temporarily adjust their foreign card payment limit before going on vacation."
]

TONES = ["urgent and hectic", "confused and hesitant", "overly friendly and distracting", "objective and professional"]
LENGTH_PROMPTS = [("short (4-6 turns)", 600), ("medium (8-12 turns)", 1000)]

class TranscriptSchema(BaseModel):
    id: str
    text: str = Field(description="Complete conversation transcript between customer and agent")
    source: str = "nemo-curator-custom"

system_prompt = (
    "You are a synthetic data generator for Banking Fraud Detection.\n"
    "Generate realistic phone transcripts between a caller and a bank agent.\n"
    "Use speaker prefixes 'Customer:' and 'Agent:'.\n"
    "Respond ONLY with a valid JSON object without Markdown formatting:\n"
    '{"id": "doc-XXX", "text": "Customer: ... \\nAgent: ...", "source": "nemo-curator-custom"}'
)

## 3. Generation & LLM-as-a-Judge Pipeline with W&B Artifact Lineage
Tracks performance (latency, throughput), data quality (judge scores, text length), logs an interactive sample table, and publishes the versioned `raw-transcripts` artifact.

In [3]:
async def test_llm_connection():
    print(f"🔍 Testing NIM endpoint at {NIM_BASE_URL}...")
    response = curator_openai_client.query_model(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": "Reply only with 'OK'"}],
        max_tokens=10
    )
    print(f"✅ NIM Connection {response[0].strip()}!")

async def evaluate_sample_quality(text: str) -> int:
    eval_prompt = (
        "Rate this bank transcript from 1-5 on realism for Fraud Detection SFT. "
        f"Transcript: {text}\nRespond ONLY with JSON: {{\"score\": int}}"
    )
    loop = asyncio.get_running_loop()
    try:
        response = await loop.run_in_executor(
            None,
            lambda: curator_openai_client.query_model(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": eval_prompt}],
                max_tokens=50
            )
        )
        clean_json = re.sub(r"^```(?:json)?\s*|\s*```$", "", response[0].strip())
        return int(json.loads(clean_json).get("score", 0))
    except Exception:
        return 0

async def generate_single_sample(index: int, f_data, f_bench, rails_app: LLMRails):
    global completed_counter
    doc_id = f"doc-{index:05d}"
    is_fraud = (index % 2 != 0)
    label = "fraud" if is_fraud else "legitimate"
    scenario = random.choice(FRAUD_SCENARIOS if is_fraud else LEGITIMATE_SCENARIOS)
    customer_name = random.choice(CUSTOMER_NAMES)
    tone = random.choice(TONES)
    length_desc, _ = random.choice(LENGTH_PROMPTS)

    user_prompt = (
        f"Generate a {'FRAUD' if is_fraud else 'LEGITIMATE'} bank call.\n"
        f"Customer: {customer_name} | Tone: {tone} | Length: {length_desc}\n"
        f"Scenario: {scenario}"
    )

    async with semaphore:
        for attempt in range(3):
            start_time = time.time()
            try:
                rails_response = await rails_app.generate_async(messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ])
                latency = time.time() - start_time
                
                raw_content = rails_response.get("content", str(rails_response)) if isinstance(rails_response, dict) else str(rails_response)
                clean_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_content, flags=re.MULTILINE).strip()
                
                try:
                    raw_json = json.loads(clean_text)
                except json.JSONDecodeError:
                    raw_json = {"id": doc_id, "text": clean_text, "source": "nemo-curator-custom"}

                data = TranscriptSchema(id=doc_id, text=raw_json.get("text", raw_content))
                score = await evaluate_sample_quality(data.text)

                if score >= 4:
                    async with file_lock:
                        f_data.write(data.model_dump_json() + "\n")
                        f_bench.write(json.dumps({"id": doc_id, "label": label, "score": score}) + "\n")
                        generated_records.append({
                            "id": doc_id,
                            "label": label,
                            "tone": tone,
                            "judge_score": score,
                            "char_length": len(data.text),
                            "transcript": data.text
                        })
                    
                    completed_counter += 1
                    text_length = len(data.text)
                    approx_tokens = text_length // 4  # Rough estimation of tokens
                    tokens_per_sec = approx_tokens / latency if latency > 0 else 0
                    
                    # Comprehensive W&B Metrics Logging
                    run.log({
                        "generation/completed_samples": completed_counter,
                        "generation/judge_score": score,
                        "generation/progress_pct": (completed_counter / NUM_SAMPLES) * 100,
                        "generation/latency_seconds": latency,
                        "generation/throughput_tokens_per_sec": tokens_per_sec,
                        "generation/transcript_length_chars": text_length,
                        "generation/is_fraud_sample": 1 if is_fraud else 0,
                        "generation/retry_count": attempt
                    })

                    if completed_counter % 5 == 0 or completed_counter == NUM_SAMPLES:
                        print(f"⏳ Progress: [{completed_counter}/{NUM_SAMPLES}] ({completed_counter/NUM_SAMPLES*100:.1f}%)")
                    return
            except Exception:
                if attempt == 2:
                    run.log({"generation/parser_errors": 1})
                await asyncio.sleep(1)

async def main():
    await test_llm_connection()
    config = RailsConfig.from_content(
        colang_content="",
        yaml_content=f"models:\n  - type: main\n    engine: openai\n    model: {GEN_MODEL}\n    parameters:\n      base_url: {NIM_BASE_URL}\n      api_key: not-needed"
    )
    rails_app = LLMRails(config)
    print(f"🚀 Generating {NUM_SAMPLES} synthetic transcripts via NeMo Guardrails...")
    os.makedirs(os.path.dirname(OUTPUT_DATA_FILE), exist_ok=True)
    
    with open(OUTPUT_DATA_FILE, "w", encoding="utf-8") as f_data, open(OUTPUT_BENCHMARK_FILE, "w", encoding="utf-8") as f_bench:
        await asyncio.gather(*[generate_single_sample(i, f_data, f_bench, rails_app) for i in range(1, NUM_SAMPLES + 1)])
    
    print(f"✅ Generation complete! Saved to {OUTPUT_DATA_FILE}")

    # Calculate dataset statistics for Artifact metadata
    fraud_count = sum(1 for r in generated_records if r["label"] == "fraud")
    legit_count = len(generated_records) - fraud_count
    avg_score = sum(r["judge_score"] for r in generated_records) / len(generated_records) if generated_records else 0

    # Log W&B Table for visual traceability
    preview_table = wandb.Table(columns=["id", "label", "tone", "judge_score", "char_length", "transcript"])
    for r in generated_records:
        preview_table.add_data(r["id"], r["label"], r["tone"], r["judge_score"], r["char_length"], r["transcript"])
    run.log({"generation/dataset_preview": preview_table})

    # Version and upload dataset as a W&B Artifact with rich lineage metadata
    print(f"📦 Uploading dataset artifact '{ARTIFACT_NAME}' to Weights & Biases...")
    artifact = wandb.Artifact(
        name=ARTIFACT_NAME,
        type="dataset",
        description="Raw synthetic banking call transcripts (Fraud vs. Legitimate) gated by LLM-as-a-Judge (>= 4/5)",
        metadata={
            "stage": "01_data_generation",
            "generator_model": GEN_MODEL,
            "judge_model": JUDGE_MODEL,
            "total_samples": len(generated_records),
            "fraud_samples": fraud_count,
            "legitimate_samples": legit_count,
            "avg_judge_score": round(avg_score, 2)
        }
    )
    artifact.add_file(OUTPUT_DATA_FILE, name="transcripts.jsonl")
    artifact.add_file(OUTPUT_BENCHMARK_FILE, name="benchmark.jsonl")
    
    logged_artifact = run.log_artifact(artifact, aliases=["latest", "raw-v1"])
    logged_artifact.wait()  # Ensure artifact is fully registered before finishing run
    
    run.finish()
    print(f"✅ W&B Lineage registered! Artifact '{ARTIFACT_NAME}:latest' is ready for Data Curation.")

await main()

🔍 Testing NIM endpoint at http://172.17.0.1:8800/v1...
✅ NIM Connection OK!


/usr/local/lib/python3.10/dist-packages/nemoguardrails/llm/frameworks/default.py:101: UserWarning: API key will be sent over plaintext HTTP to http://172.17.0.1:8800/v1; use https:// for production deployments.
  self._clients[key] = OpenAICompatibleClient(base_url=base_url, api_key=api_key, **client_kwargs)


🚀 Generating 100 synthetic transcripts via NeMo Guardrails...
⏳ Progress: [5/100] (5.0%)
⏳ Progress: [10/100] (10.0%)
⏳ Progress: [15/100] (15.0%)
⏳ Progress: [20/100] (20.0%)
⏳ Progress: [25/100] (25.0%)
⏳ Progress: [30/100] (30.0%)
⏳ Progress: [35/100] (35.0%)
⏳ Progress: [40/100] (40.0%)
⏳ Progress: [45/100] (45.0%)
⏳ Progress: [50/100] (50.0%)
⏳ Progress: [55/100] (55.0%)
⏳ Progress: [60/100] (60.0%)
⏳ Progress: [65/100] (65.0%)
⏳ Progress: [70/100] (70.0%)
⏳ Progress: [75/100] (75.0%)
⏳ Progress: [80/100] (80.0%)
⏳ Progress: [85/100] (85.0%)
⏳ Progress: [90/100] (90.0%)
✅ Generation complete! Saved to /data/nemo-fraud-detection-notebooks/notebooks/01_Data_Generation/data/transcripts.jsonl
📦 Uploading dataset artifact 'raw-transcripts' to Weights & Biases...


generation/completed_samples,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇████
generation/is_fraud_sample,█▁▁▁▁█▁█▁█▁▁▁██▁▁▁██▁█▁███▁██▁▁█▁██▁▁█▁█
generation/judge_score,▁▁▁█▁▁▁▁▁▁█▁▁▁▁▁▁▁█▁▁█▁▁▁▁█▁▁▁██▁▁▁██▁█▁
generation/latency_seconds,▂▄▃▃▅▇▃▅▃▁▇▆▂▃▄▂▄▄▁▃▅▆▂▅▇▄▁▁▅▃▄▁▂▅▅▄█▁▂▂
generation/progress_pct,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇████
generation/retry_count,▁▁▁▅▁▁█▁▅▁▁▁▁▁▁▁▁█▁▁▁▅▁▁▁▁▅▁▁▅▅▁▁▁▁▁▁▁▅▅
generation/throughput_tokens_per_sec,▃▁▃▄█▃▃▄▅▄▃▄▂▃▅▅▂▃▄▄▃▄▂▃▃▃▅▄▂▂▄▅▃▃▄▄▆▄▃▃
generation/transcript_length_chars,▂▄▁▄▄▆▆▅▂▃▂▁▄▁▃▂▄▃▄▂▁█▄▂▂▁▃▄▁▅▂▇▃▃▁▃▇▅▄▄
generation/completed_samples,93
generation/is_fraud_sample,1
generation/judge_score,4


✅ W&B Lineage registered! Artifact 'raw-transcripts:latest' is ready for Data Curation.


## 4. Live Output Preview
Inspecting generated samples directly from `transcripts.jsonl` and `benchmark.jsonl`.

In [4]:
with open(OUTPUT_DATA_FILE, "r", encoding="utf-8") as f_d, open(OUTPUT_BENCHMARK_FILE, "r", encoding="utf-8") as f_b:
    samples = [json.loads(line) for line in f_d if line.strip()[:2] == "{"]
    benchmarks = {json.loads(line)["id"]: json.loads(line) for line in f_b if line.strip()[:2] == "{"}

for s in samples[:2]:
    meta = benchmarks.get(s["id"], {})
    print(f"--- SAMPLE | ID: {s['id']} | LABEL: {meta.get('label', 'N/A').upper()} | JUDGE SCORE: {meta.get('score', 'N/A')}/5 ---")
    print(s["text"] + "\n")

### Key Takeaways
* **End-to-End Traceability:** Registered as `job_type="data-generation"` producing the versioned `raw-transcripts` artifact.
* **Auditability:** Full dataset preview table and class distribution stored directly in W&B Artifact metadata.